<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/Surname_Classification_with_RNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Pre-Processing

In [1]:
import pandas as pd
import numpy as np
import collections
import re
from argparse import Namespace

In [2]:
#Import Dataset
import kagglehub
path = kagglehub.dataset_download('alenic/surname-dataset-classification')
print("Path to dataset files:", path)

100%|██████████| 307k/307k [00:00<00:00, 88.0MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/alenic/surname-dataset-classification/versions/2


In [3]:
!mv '/root/.cache/kagglehub/datasets/alenic/surname-dataset-classification/versions/2' '/content/'

In [4]:
args = Namespace(
    raw_dataset_csv = '/content/2/surname-nationality.csv',
    train_proportion = 0.7,
    val_proportion = 0.15,
    test_proportion = 0.15,
    output_munged_csv = '/content/surnames_with_splits.csv',
    seed = 1337
)

In [5]:
data = pd.read_csv(args.raw_dataset_csv,header=0)

In [6]:
data.head()

,rank,surname,nationality,frequency,census_count
0,1.0,Tesfaye,Ethiopian,0.011905,1167260
1,2.0,Mohammed,Ethiopian,0.011111,1084839
2,3.0,Getachew,Ethiopian,0.009174,895366
3,4.0,Abebe,Ethiopian,0.008475,825501
4,5.0,Girma,Ethiopian,0.008403,822765


In [7]:
surnames = data[['surname','nationality']]

In [8]:
surnames.head()

,surname,nationality
0,Tesfaye,Ethiopian
1,Mohammed,Ethiopian
2,Getachew,Ethiopian
3,Abebe,Ethiopian
4,Girma,Ethiopian


In [9]:
len(set(surnames.nationality))

37

In [10]:
#splitting data by nationality
by_nationality  = collections.defaultdict(list)
for _,row in surnames.iterrows():
  by_nationality[row.nationality].append(row.to_dict())

In [11]:
#Split the data
final_list=[]
np.random.seed(args.seed)

for _,item_list in sorted(by_nationality.items()):
  np.random.shuffle(item_list)
  n=len(item_list)
  n_train = int(args.train_proportion*n)
  n_val = int(args.val_proportion*n)
  n_test = int(args.test_proportion*n)
  # print(type(item_list))
  # print(type(item_list[0]))
  for item in item_list[:n_train]:
    item['split']='train'
  for item in item_list[n_train:n_train+n_val]:
    item['split']='val'
  for item in item_list[n_train+n_val:]:
    item['split']='test'

  final_list.extend(item_list)

In [12]:
final_surnames = pd.DataFrame(final_list)

In [13]:
final_surnames.split.value_counts()

,count
split,
train,25358
test,5460
val,5423


In [14]:
final_surnames.to_csv(args.output_munged_csv,index=False)

Surname Classification RNNs

In [15]:
from argparse import Namespace
import os
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [17]:
#Vocabulary Class
class Vocabulary(object):
  #Class to process test and extract vocabulary for mapping
  def __init__(self,token_to_idx=None,add_unk=True,unk_token="<UNK"):
    #token_to_idx - dict - a map for token to indices
    if token_to_idx is None:
      token_to_idx={}
    self._token_to_idx = token_to_idx

    self._idx_to_token = {idx:token for token,idx in self._token_to_idx.items()}

  def to_serializable(self):
    return {'token_to_idx':self._token_to_idx}

  @classmethod
  def from_serializable(cls,contents):
    return cls(**contents)

  def add_token(self,token):
    if token in self._token_to_idx:
        index = self._token_to_idx[token]
    else:
        index = len(self._token_to_idx)
        self._token_to_idx[token] = index
        self._idx_to_token[index] = token
    return index

  def add_many(self,tokens):
    return [self.add_token(token) for token in tokens]

  def lookup_token(self,token):
    return self._token_to_idx[token]

  def lookup_index(self,index):
    if index not in self._idx_to_token:
      raise KeyError("the index (%d) is not in the Vocabulary" % index)
    return self._idx_to_token[index]

  def __str__(self) -> str:
    return "<Vocabulary(size=%d)>" % len(self)

  def __len__(self):
    return len(self._token_to_idx)

In [18]:
class SequenceVocabulary(Vocabulary):
  def __init__(self,token_to_idx=None,unk_token="<UNK>",mask_token="<MASK>",begin_seq_token="<BEGIN>",end_seq_token="<END>"):
    super(SequenceVocabulary,self).__init__(token_to_idx)

    self._mask_token = mask_token
    self._unk_token = unk_token
    self._begin_seq_token = begin_seq_token
    self._end_seq_token = end_seq_token

    self.mask_index = self.add_token(self._mask_token)
    self.unk_index = self.add_token(self._unk_token)
    self.begin_seq_token = self.add_token(self._begin_seq_token)
    self.end_seq_token = self.add_token(self._end_seq_token)

  def to_serializable(self):
    contents = super(SequenceVocabulary,self).to_serializable()
    contents.update({'unk_token':self._unk_token,
                     'mask_token':self._mask_token,
                     'begin_seq_token':self._begin_seq_token,
                     'end_seq_token':self._end_seq_token})
    return contents

  def lookup_token(self, token):
    if self.unk_index >= 0:
        return self._token_to_idx.get(token, self.unk_index)
    else:
        return self._token_to_idx[token]

In [37]:
#Vecotrizer Class
class SurnameVectorizer(object):
  #Coordinates Vocabularies and put them to use
  def __init__(self,char_vocab,nationality_vocab):
    self.char_vocab = char_vocab
    self.nationality_vocab = nationality_vocab

  def vectorize(self,surname,vector_lenght=-1):
    indices = [self.char_vocab.begin_seq_token]
    indices.extend(self.char_vocab.lookup_token(token) for token in surname)
    indices.append(self.char_vocab.end_seq_token)
    if vector_lenght < 0:
      vector_lenght = len(indices)
    out_vector = np.zeros(vector_lenght,dtype=np.int64)
    out_vector[:len(indices)] = indices
    out_vector[len(indices):] = self.char_vocab.mask_index
    return out_vector,len(indices)

  @classmethod
  def from_dataframe(cls,surname_df):
    char_vocab = SequenceVocabulary()
    nationality_vocab = Vocabulary()
    for index,row in surname_df.iterrows():
      for char in row.surname:
        char_vocab.add_token(char)
      nationality_vocab.add_token(row.nationality)
    return cls(char_vocab,nationality_vocab)

  @classmethod
  def from_serializable(cls,contents):
    char_vocab = SequenceVocabulary.from_serializable(contents['char_vocab'])
    nat_vocab = Vocabulary.from_serializable(contents['nationality_vocab'])
    return cls(char_vocab=char_vocab,nationality_vocab=nat_vocab)

  def to_serializable(self):
    return {'char_vocab':self.char_vocab.to_serializable(),
            'nationality_vocab':self.nationality_vocab.to_serializable()}

In [38]:
class SurnameDataset(Dataset):
  def __init__(self,surname_df,vectorizer):
    self.surname_df = surname_df
    self._vectorizer = vectorizer
    self._max_seq_length = max(map(len,self.surname_df.surname)) + 2
    self.train_df = self.surname_df[self.surname_df['split']=='train']
    self.train_size = len(self.train_df)

    self.val_df = self.surname_df[self.surname_df['split']=='val']
    self.validation_size = len(self.val_df)

    self.test_df = self.surname_df[self.surname_df['split']=='test']
    self.test_size = len(self.test_df)

    self.lookup_dict = {'train':(self.train_df,self.train_size),
                         'val':(self.val_df,self.validation_size),
                         'test':(self.test_df,self.test_size)}
    self.set_split('train')

    class_counts = surname_df.nationality.value_counts().to_dict()
    def sort_key(item):
      return self._vectorizer.nationality_vocab.lookup_token(item[0])
    sorted_counts = sorted(class_counts.items(),key=sort_key)
    frequencies = [count for _,count in sorted_counts]
    self.class_weights = 1.0 / torch.tensor(frequencies,dtype=torch.float32)

  @classmethod
  def load_dataset_and_make_vectorizer(cls,surname_csv):
    surname_df = pd.read_csv(surname_csv)
    train_surname_df = surname_df[surname_df['split']=='train']
    return cls(surname_df,SurnameVectorizer.from_dataframe(train_surname_df))

  @classmethod
  def load_dataset_and_load_vectorizer(cls,surname_csv,vectorizer_filepath):
    surname_df = pd.read_csv(surname_csv)
    vectorizer = cls.load_vectorizer_only(vectorizer_filepath)
    return cls(surname_df,vectorizer)

  @classmethod
  def load_vectorizer_only(cls,vectorizer_filepath):
    with open(vectorizer_filepath) as fp:
      return SurnameVectorizer.from_serializable(json.load(fp))

  def save_vectorizer(self,vectorizer_filepath):
    with open(vectorizer_filepath,'w') as fp:
      json.dump(self._vectorizer.to_serializable(),fp)

  def get_vectorizer(self):
    return self._vectorizer

  def set_split(self,split='train'):
    self._target_split = split
    self._target_df,self._target_size = self.lookup_dict[split]

  def __getitem__(self, index):
    row = self._target_df.iloc[index]
    surname_vector,vec_lenght = self._vectorizer.vectorize(row.surname,self._max_seq_length)
    nationality_index= self._vectorizer.nationality_vocab.lookup_token(row.nationality)
    return {'x_data':surname_vector,
            'y_target':nationality_index,
            'x_length':vec_lenght}

  def __len__(self):
        return self._target_size

  def get_num_batches(self,batch_size):
    return len(self) // batch_size

In [39]:
def generator_batches(dataset,batch_size,shuffle=True,drop_last=True,device='cpu'):
  dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  for data_dict in dataloader:
    out_data_dict = {}
    for name,tensor in data_dict.items():
      out_data_dict[name] = data_dict[name].to(device)
    yield out_data_dict

In [22]:
def column_gather(y_out,x_lenghts):
  x_lenghts = x_lenghts.long().detach().cpu().numpy()-1
  out=[]
  for batch_index,column_index in enumerate(x_lenghts):
    out.append(y_out[batch_index,column_index])
  return torch.stack(out)

In [23]:
class ElmanRNN(nn.Module):
  def __init__(self,input_size,hidden_size,batch_first=False):
    super(ElmanRNN,self).__init__()
    self.rnn_cell = nn.RNNCell(input_size,hidden_size)
    self.batch_first = batch_first
    self.hidden_size = hidden_size

  def _initial_hidden(self,batch_size):
    return torch.zeros((batch_size,self.hidden_size))

  def forward(self,x_in,initial_hidden=None):
    if self.batch_first:
      batch_size,seq_size,feat_size = x_in.size()
      x_in = x_in.permute(1,0,2)
    else:
      seq_size,batch_size,feat_size = x_in.size()

    hiddens =[]
    if initial_hidden is None:
      initial_hidden = self._initial_hidden(batch_size)
      initial_hidden = initial_hidden.to(x_in.device)
    hidden_t = initial_hidden
    for t in range(seq_size):
      hidden_t = self.rnn_cell(x_in[t],hidden_t)
      hiddens.append(hidden_t)

    hiddens = torch.stack(hiddens)
    if self.batch_first:
      hiddens = hiddens.permute(1,0,2)
    return hiddens

In [24]:
class SurnameClassifier(nn.Module):
  def __init__(self,embedding_size,num_embeddings,num_classes,rnn_hidden_size,batch_first=True,padding_idx=0):
    super(SurnameClassifier,self).__init__()
    self.emb = nn.Embedding(num_embeddings=num_embeddings,embedding_dim=embedding_size,padding_idx=padding_idx)
    self.rnn = ElmanRNN(input_size=embedding_size,hidden_size=rnn_hidden_size,batch_first=batch_first)
    self.fc1 = nn.Linear(in_features=rnn_hidden_size,out_features=rnn_hidden_size)
    self.fc2 = nn.Linear(in_features=rnn_hidden_size,out_features=num_classes)
  def forward(self,x_in,x_lengths=None,apply_softmax=False):
    x_embedded = self.emb(x_in)
    y_out = self.rnn(x_embedded)
    if x_lengths is not None:
      y_out = column_gather(y_out,x_lengths)
    else:
      y_out = y_out[:,-1,:]
    y_out = F.relu(self.fc1(F.dropout(y_out,0.5)))
    y_out = self.fc2(F.dropout(y_out,0.5))

    if apply_softmax:
      y_out= F.softmax(y_out,dim=1)
    return y_out

In [25]:
def set_seed_everywhere(seed,cuda):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if cuda:
    torch.cuda.manual_seed_all(seed)

In [26]:
def handle_dirs(dirpath):
  if not os.path.exists(dirpath):
    os.makedirs(dirpath)

In [27]:
args = Namespace(
    surname_csv = '/content/surnames_with_splits.csv',
    vectorizer_file = 'vectorizer.json',
    model_state_file = 'model.pth',
    save_dir = '/content/model_storage/',
    char_embedding_size=100,
    rnn_hidden_size=64,
    seed=1337,
    num_epochs=100,
    early_stopping_criteria=5,
    learning_rate=0.001,
    batch_size=64,
    cuda=True,
    reload_from_files=False,
    expand_filepaths_to_save_dir=True,
    catch_keyboard_interrupt=True,
)

In [28]:
if args.expand_filepaths_to_save_dir:
  args.vectorizer_file = os.path.join(args.save_dir,args.vectorizer_file)
  args.model_state_file = os.path.join(args.save_dir,args.model_state_file)
  print("Expanded filepaths: ")
  print("\t{}".format(args.vectorizer_file))
  print("\t{}".format(args.model_state_file))

Expanded filepaths: 
	/content/model_storage/vectorizer.json
	/content/model_storage/model.pth


In [29]:
if not torch.cuda.is_available():
  args.cuda=False
args.device = torch.device("cuda" if args.cuda else "cpu")
print(args.device)

cuda


In [30]:
set_seed_everywhere(args.seed, args.cuda)
handle_dirs(args.save_dir)

In [40]:
if args.reload_from_files:
  print('Reloading')
  dataset = SurnameDataset.load_dataset_and_load_vectorizer(args.surname_csv,args.vectorizer_file)
else:
  print('Creating')
  dataset = SurnameDataset.load_dataset_and_make_vectorizer(args.surname_csv)
  dataset.save_vectorizer(args.vectorizer_file)

vectorizer = dataset.get_vectorizer()
classifier = SurnameClassifier(embedding_size=args.char_embedding_size,
                               num_embeddings=len(vectorizer.char_vocab),
                               num_classes=len(vectorizer.nationality_vocab),
                               rnn_hidden_size=args.rnn_hidden_size,
                               padding_idx=vectorizer.char_vocab.mask_index)

Creating


In [41]:
def make_train_state(args):
  return {
      'stop_early':False,
      'early_stopping_step':0,
      'early_stopping_best_val':1e8,
      'learning_rate':args.learning_rate,
      'epoch_index':0,
      'train_loss':[],
      'train_acc':[],
      'val_loss':[],
      'val_acc':[],
      'test_loss':-1,
      'test_acc':-1,
      'model_filename':args.model_state_file
       }

In [42]:
def update_train_state(args, model, train_state):
    # Save one model at least
    if train_state['epoch_index'] == 0:
        torch.save(model.state_dict(), train_state['model_filename'])
        train_state['stop_early'] = False

    # Save model if performance improved
    elif train_state['epoch_index'] >= 1:
        loss_tm1, loss_t = train_state['val_loss'][-2:]

        # If loss worsened
        if loss_t >= loss_tm1:
            # Update step
            train_state['early_stopping_step'] += 1
        # Loss decreased
        else:
            # Save the best model
            if loss_t < train_state['early_stopping_best_val']:
                torch.save(model.state_dict(), train_state['model_filename'])
                train_state['early_stopping_best_val'] = loss_t

            # Reset early stopping step
            train_state['early_stopping_step'] = 0

        # Stop early ?
        train_state['stop_early'] = \
            train_state['early_stopping_step'] >= args.early_stopping_criteria

    return train_state

In [34]:
def compute_accuracy(y_pred, y_target):
    _, y_pred_indices = y_pred.max(dim=1)
    n_correct = torch.eq(y_pred_indices, y_target).sum().item()
    return n_correct / len(y_pred_indices) * 100

In [43]:
classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)

loss_func = nn.CrossEntropyLoss(dataset.class_weights)
optimizer = optim.Adam(classifier.parameters(), lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,
                                           mode='min', factor=0.5,
                                           patience=1)

train_state = make_train_state(args)

In [44]:
try:
    for epoch_index in range(args.num_epochs):
        train_state['epoch_index'] = epoch_index
        dataset.set_split('train')
        batch_generator = generator_batches(dataset=dataset,
                                           batch_size=args.batch_size,
                                           device=args.device)
        running_loss = 0.0
        running_acc = 0.0
        classifier.train()
        for batch_index, batch_dict in enumerate(batch_generator):
            optimizer.zero_grad()
            y_pred = classifier(x_in=batch_dict['x_data'],x_lengths=batch_dict['x_length'])
            loss = loss_func(y_pred,batch_dict['y_target'])
            running_loss+=(loss.item()-running_loss)/(batch_index+1)
            loss.backward()
            optimizer.step()
            acc_t = compute_accuracy(y_pred,batch_dict['y_target'])
            running_acc+=(acc_t-running_acc)/(batch_index+1)
        train_state['train_loss'].append(running_loss)
        train_state['train_acc'].append(running_acc)
        dataset.set_split('val')
        batch_generator = generator_batches(dataset=dataset,
                                           batch_size=args.batch_size,
                                           device=args.device)
        running_loss = 0.0
        running_acc = 0.0
        classifier.eval()
        for batch_index, batch_dict in enumerate(batch_generator):
            y_pred = classifier(x_in=batch_dict['x_data'],x_lengths=batch_dict['x_length'])
            loss = loss_func(y_pred,batch_dict['y_target'])
            running_loss+=(loss.item()-running_loss)/(batch_index+1)
            acc_t = compute_accuracy(y_pred,batch_dict['y_target'])
            running_acc+=(acc_t-running_acc)/(batch_index+1)
        train_state['val_acc'].append(running_acc)
        train_state['val_loss'].append(running_loss)
        train_state = update_train_state(args=args, model=classifier,train_state=train_state)
        scheduler.step(train_state['val_loss'][-1])
        print("Epoch: {}/{}...".format(epoch_index+1,args.num_epochs),
              "Train Loss: {:.4f}".format(train_state['train_loss'][-1]),
              "Train Acc: {:.2f}".format(train_state['train_acc'][-1]),
              "Val Loss: {:.4f}".format(train_state['val_loss'][-1]),
              "Val Acc: {:.2f}".format(train_state['val_acc'][-1]))
        if train_state['stop_early']:
            break
except KeyboardInterrupt:
  print('Exiting')



Epoch: 1/100... Train Loss: 3.4911 Train Acc: 13.01 Val Loss: 3.2780 Val Acc: 21.24
Epoch: 2/100... Train Loss: 3.1145 Train Acc: 22.66 Val Loss: 2.9987 Val Acc: 23.18
Epoch: 3/100... Train Loss: 2.8929 Train Acc: 22.68 Val Loss: 2.8715 Val Acc: 23.46
Epoch: 4/100... Train Loss: 2.7858 Train Acc: 23.69 Val Loss: 2.7605 Val Acc: 24.26
Epoch: 5/100... Train Loss: 2.7231 Train Acc: 23.91 Val Loss: 2.7514 Val Acc: 23.33
Epoch: 6/100... Train Loss: 2.6765 Train Acc: 24.70 Val Loss: 2.7177 Val Acc: 24.68
Epoch: 7/100... Train Loss: 2.6405 Train Acc: 25.49 Val Loss: 2.6788 Val Acc: 27.10
Epoch: 8/100... Train Loss: 2.5977 Train Acc: 26.09 Val Loss: 2.6666 Val Acc: 25.63
Epoch: 9/100... Train Loss: 2.5773 Train Acc: 25.59 Val Loss: 2.6947 Val Acc: 25.56
Epoch: 10/100... Train Loss: 2.5433 Train Acc: 26.56 Val Loss: 2.6533 Val Acc: 25.67
Epoch: 11/100... Train Loss: 2.5303 Train Acc: 27.28 Val Loss: 2.6560 Val Acc: 26.04
Epoch: 12/100... Train Loss: 2.5001 Train Acc: 27.07 Val Loss: 2.6221 Val 

In [46]:
classifier.load_state_dict(torch.load(train_state['model_filename']))

classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)
loss_func = nn.CrossEntropyLoss(dataset.class_weights)

dataset.set_split('test')
batch_generator = generator_batches(dataset,
                                   batch_size=args.batch_size,
                                   device=args.device)
running_loss = 0.
running_acc = 0.
classifier.eval()

for batch_index, batch_dict in enumerate(batch_generator):
    # compute the output
    y_pred =  classifier(batch_dict['x_data'],
                         x_lengths=batch_dict['x_length'])

    # compute the loss
    loss = loss_func(y_pred, batch_dict['y_target'])
    loss_t = loss.item()
    running_loss += (loss_t - running_loss) / (batch_index + 1)

    # compute the accuracy
    acc_t = compute_accuracy(y_pred, batch_dict['y_target'])
    running_acc += (acc_t - running_acc) / (batch_index + 1)

train_state['test_loss'] = running_loss
train_state['test_acc'] = running_acc
print("Test loss: {};".format(train_state['test_loss']))
print("Test Accuracy: {}".format(train_state['test_acc']))

Test loss: 2.570014653486364;
Test Accuracy: 28.345588235294116
